# Recovery Notebook — Regenerate Plots & pred.npz from Drive
Run cells **top to bottom**. Each cell saves to Drive before proceeding.
Weights and `history.json` are read directly from Drive — no re-training needed.


In [ ]:
# 0. Check GPU
!nvidia-smi


In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 2. Configure paths — edit DRIVE_ROOT if your files are in a subfolder
from pathlib import Path
import shutil

DRIVE_ROOT   = Path('/content/drive/MyDrive/vis_hw4_checkpoints')  # where weights were saved
DRIVE_OUT    = Path('/content/drive/MyDrive/vis_hw4_outputs')       # where results will be saved
WORK_DIR     = Path('/content/vis_hw4')

for d in [WORK_DIR/'checkpoints', WORK_DIR/'plots', WORK_DIR/'predictions',
          WORK_DIR/'dataset', DRIVE_OUT/'plots', DRIVE_OUT/'predictions']:
    d.mkdir(parents=True, exist_ok=True)

# Copy checkpoints + history from Drive into working directory
for fname in ['stage1_best.pth', 'stage2_best.pth', 'history.json']:
    src = DRIVE_ROOT / fname
    dst = WORK_DIR / 'checkpoints' / fname
    if src.exists():
        shutil.copy2(src, dst)
        print(f'Copied {fname}')
    else:
        print(f'WARNING: {fname} not found at {src}')

# Also copy train.py from Drive
for src_path in [DRIVE_ROOT / 'train.py',
                 Path('/content/drive/MyDrive/train.py')]:
    if src_path.exists():
        shutil.copy2(src_path, WORK_DIR / 'train.py')
        print(f'Copied train.py from {src_path}')
        break
else:
    print('WARNING: train.py not found on Drive — upload it manually')


In [ ]:
# 3. Copy dataset zip and extract (needed for inference)
from pathlib import Path
import shutil

zip_dest = Path('/content/vis_hw4/hw4_realse_dataset.zip')
ext_dir  = Path('/content/vis_hw4/dataset')

# Look for zip on Drive
for candidate in [
    Path('/content/drive/MyDrive/hw4_realse_dataset.zip'),
    Path('/content/drive/MyDrive/vis_hw4_checkpoints/hw4_realse_dataset.zip'),
    Path('/content/hw4_realse_dataset.zip'),
]:
    if candidate.exists():
        shutil.copy2(candidate, zip_dest)
        print(f'Copied zip from {candidate}')
        break
else:
    print('WARNING: zip not found — upload hw4_realse_dataset.zip to /content/ or Drive')

# Extract if not already done
if zip_dest.exists() and not any(ext_dir.rglob('*.png')):
    import zipfile
    print('Extracting...')
    with zipfile.ZipFile(zip_dest) as zf:
        zf.extractall(ext_dir)
    print('Extracted.')
elif any(ext_dir.rglob('*.png')):
    print('Dataset already extracted.')


In [ ]:
# 4. Verify history.json content
import json
from pathlib import Path

hist = json.load(open('/content/vis_hw4/checkpoints/history.json'))
for stage, data in hist.items():
    n = len(data['total'])
    best = max(data['total'])
    best_r = max(data['rain'])
    best_s = max(data['snow'])
    print(f'{stage}: {n} epochs  |  best total={best:.4f}  rain={best_r:.4f}  snow={best_s:.4f}')


In [ ]:
# 5. Regenerate all 6 plots → saved to Drive immediately
!pip install -q matplotlib

import json, math
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shutil

hist      = json.load(open('/content/vis_hw4/checkpoints/history.json'))
plot_dir  = Path('/content/vis_hw4/plots')
DRIVE_OUT = Path('/content/drive/MyDrive/vis_hw4_outputs')
(DRIVE_OUT / 'plots').mkdir(parents=True, exist_ok=True)

COLOURS = {'stage1': '#2196F3', 'stage2': '#FF5722'}
LABELS  = {'stage1': 'Stage 1 — Base Training (180 ep)',
            'stage2': 'Stage 2 — Fine-tune (40 ep)'}

def save(fig, name):
    local = plot_dir / name
    drive = DRIVE_OUT / 'plots' / name
    fig.savefig(local, dpi=150, bbox_inches='tight')
    shutil.copy2(local, drive)
    plt.close(fig)
    print(f'Saved {name} → Drive')

# ── Per-stage PSNR & Loss ────────────────────────────────────────────────
for sname, data in hist.items():
    eps = range(1, len(data['total']) + 1)
    col = COLOURS.get(sname, '#333')
    lbl = LABELS.get(sname, sname)

    # PSNR
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(eps, data['total'], color=col,       label='Total',  linewidth=2)
    ax.plot(eps, data['rain'],  color='#E91E63',  label='Rain',   linewidth=1.5, linestyle='--')
    ax.plot(eps, data['snow'],  color='#00BCD4',  label='Snow',   linewidth=1.5, linestyle=':')
    ax.set_title(lbl, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('PSNR (dB)')
    lo = max(0, min(min(data['rain']), min(data['snow']), min(data['total'])) - 0.5)
    hi = max(max(data['rain']), max(data['snow']), max(data['total'])) + 0.3
    ax.set_ylim(lo, hi); ax.legend(loc='lower right'); ax.grid(alpha=0.3)
    plt.tight_layout()
    save(fig, f'psnr_{sname}.png')

    # Loss
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(eps, data['loss'], color=col, linewidth=1.8)
    ax.set_title(lbl, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Charbonnier Loss'); ax.grid(alpha=0.3)
    plt.tight_layout()
    save(fig, f'loss_{sname}.png')

# ── Combined PSNR ────────────────────────────────────────────────────────
all_rain, all_snow, all_tot, all_loss = [], [], [], []
boundaries = [0]
stage_names = list(hist.keys())
for sname in stage_names:
    d = hist[sname]
    all_rain.extend(d['rain']); all_snow.extend(d['snow'])
    all_tot.extend(d['total']); all_loss.extend(d['loss'])
    boundaries.append(len(all_tot))

bg_colours = {'stage1': '#E3F2FD', 'stage2': '#FBE9E7'}

fig, ax = plt.subplots(figsize=(12, 5))
x = range(1, len(all_tot) + 1)
ax.plot(x, all_tot,  color='#333333', label='Total', linewidth=2)
ax.plot(x, all_rain, color='#E91E63', label='Rain',  linewidth=1.5, linestyle='--')
ax.plot(x, all_snow, color='#00BCD4', label='Snow',  linewidth=1.5, linestyle=':')
for i, sname in enumerate(stage_names):
    x0, x1 = boundaries[i] + 1, boundaries[i + 1]
    ax.axvspan(x0, x1, alpha=0.25, color=bg_colours.get(sname, '#F5F5F5'))
    ax.text((x0 + x1) / 2, max(all_tot) + 0.05,
            LABELS.get(sname, sname).split('—')[0].strip(),
            ha='center', va='bottom', fontsize=9, color=COLOURS.get(sname, '#555'))
ax.set_xlabel('Epoch (cumulative)'); ax.set_ylabel('PSNR (dB)')
ax.set_title('Full Training — PSNR Across All Stages', fontsize=13, fontweight='bold')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout()
save(fig, 'psnr_final_combined.png')

# ── Combined Loss ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(range(1, len(all_loss) + 1), all_loss, color='#37474F', linewidth=1.6)
for i, sname in enumerate(stage_names):
    x0, x1 = boundaries[i] + 1, boundaries[i + 1]
    ax.axvspan(x0, x1, alpha=0.25, color=bg_colours.get(sname, '#F5F5F5'))
    ax.text((x0 + x1) / 2, max(all_loss) * 0.98,
            LABELS.get(sname, sname).split('—')[0].strip(),
            ha='center', va='top', fontsize=9, color=COLOURS.get(sname, '#555'))
ax.set_xlabel('Epoch (cumulative)'); ax.set_ylabel('Charbonnier Loss')
ax.set_title('Full Training — Loss Across All Stages', fontsize=13, fontweight='bold')
ax.grid(alpha=0.3)
plt.tight_layout()
save(fig, 'loss_final_combined.png')

print('\nAll plots saved to Drive ✓')


In [ ]:
# 6. Display plots inline to verify
from IPython.display import Image, display
from pathlib import Path

plot_dir = Path('/content/vis_hw4/plots')
plots = [
    ('psnr_stage1.png',         'Stage 1 — PSNR (Rain / Snow / Total)'),
    ('loss_stage1.png',         'Stage 1 — Charbonnier Loss'),
    ('psnr_stage2.png',         'Stage 2 — PSNR (Rain / Snow / Total)'),
    ('loss_stage2.png',         'Stage 2 — Charbonnier Loss'),
    ('psnr_final_combined.png', 'Final — PSNR across all stages'),
    ('loss_final_combined.png', 'Final — Loss across all stages'),
]
for fname, title in plots:
    p = plot_dir / fname
    if p.exists():
        print(f'\n── {title} ──')
        display(Image(filename=str(p)))
    else:
        print(f'MISSING: {fname}')


In [ ]:
# 7. Run inference → pred.npz saved to Drive immediately
# Loads stage2_best.pth (falls back to stage1_best.pth if not present)
import torch, numpy as np
from pathlib import Path
from PIL import Image as PILImage
import torchvision.transforms.functional as TF
import shutil, sys

sys.path.insert(0, '/content/vis_hw4')
from train import PromptIR, find_pairs, list_images, basename_map

DRIVE_OUT  = Path('/content/drive/MyDrive/vis_hw4_outputs')
CKPT_DIR   = Path('/content/vis_hw4/checkpoints')
PRED_DIR   = Path('/content/vis_hw4/predictions')
EXT_DIR    = Path('/content/vis_hw4/dataset')
(DRIVE_OUT / 'predictions').mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# Load model
model = PromptIR(dim=64, prompt_len=8).to(device)
ckpt_path = CKPT_DIR / 'stage2_best.pth'
if not ckpt_path.exists():
    ckpt_path = CKPT_DIR / 'stage1_best.pth'
print(f'Loading {ckpt_path}')
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt['ema'])
model.eval()
print('Model loaded.')

# Find test images
_, test_paths = find_pairs(EXT_DIR)
print(f'Found {len(test_paths)} test images')

@torch.no_grad()
def tta_predict(model, x):
    preds = []
    for hf in [False, True]:
        for vf in [False, True]:
            xi = x
            if hf: xi = xi.flip(-1)
            if vf: xi = xi.flip(-2)
            p = model(xi)
            if vf: p = p.flip(-2)
            if hf: p = p.flip(-1)
            preds.append(p)
    return torch.stack(preds).mean(0)

# Run inference
images_dict = {}
for i, p in enumerate(test_paths):
    img = TF.to_tensor(PILImage.open(p).convert('RGB')).unsqueeze(0).to(device)
    pred = tta_predict(model, img).squeeze(0).clamp(0, 1)
    arr = (pred.cpu().numpy() * 255).round().astype(np.uint8)  # (3, H, W)
    images_dict[p.name] = arr
    if (i + 1) % 10 == 0:
        print(f'  {i+1}/{len(test_paths)}')

# Save pred.npz locally then copy to Drive immediately
npz_local = PRED_DIR / 'pred.npz'
np.savez(npz_local, **images_dict)
npz_drive = DRIVE_OUT / 'predictions' / 'pred.npz'
shutil.copy2(npz_local, npz_drive)
print(f'\npred.npz ({len(images_dict)} images) saved to Drive ✓')
print(f'  → {npz_drive}')


In [ ]:
# 8. Verify pred.npz
import numpy as np
from pathlib import Path

data = np.load('/content/vis_hw4/predictions/pred.npz')
keys = sorted(data.files, key=lambda k: int(Path(k).stem))
print(f'Total images: {len(keys)}')
print(f'Keys (first 5): {keys[:5]}')
k = keys[0]
print(f'Shape of {k}: {data[k].shape}  dtype: {data[k].dtype}')
print('All shapes uniform:', len(set(str(data[k].shape) for k in keys)) == 1)


In [ ]:
# 9. Save everything to Drive as a zip (final backup)
import shutil
from pathlib import Path

DRIVE_OUT = Path('/content/drive/MyDrive/vis_hw4_outputs')
zip_path  = Path('/content/drive/MyDrive/vis_hw4_full_backup')
shutil.make_archive(str(zip_path), 'zip', '/content/vis_hw4')
print(f'Full backup saved to {zip_path}.zip ✓')


## Drive output layout
```
MyDrive/
├── vis_hw4_checkpoints/         ← your existing weights
│   ├── stage1_best.pth
│   ├── stage2_best.pth
│   └── history.json
└── vis_hw4_outputs/
    ├── plots/
    │   ├── psnr_stage1.png
    │   ├── loss_stage1.png
    │   ├── psnr_stage2.png
    │   ├── loss_stage2.png
    │   ├── psnr_final_combined.png
    │   └── loss_final_combined.png
    └── predictions/
        └── pred.npz             ← (3,H,W) uint8, keyed by filename
```

Each cell copies to Drive **before** moving on, so a disconnect at any point leaves Drive intact.
